# Sesión 6 — Árboles y bosques: muchos árboles mejor que uno

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 2 (cierre) · **se entrega E2**

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Explicar por qué un árbol profundo **sobreajusta** y cómo un **bosque aleatorio** lo corrige promediando árboles distintos.
2. Comparar modelos con **F1** y con la **curva ROC / AUC**, y decir cuándo usar cada una.
3. Interpretar un bosque: **importancia de variables** y la opinión de sus árboles sobre un caso.
4. Detectar una **columna del futuro** en un problema de clasificación real.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                                       ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: el modelo más usado en la industria para datos de tabla, y cómo compararlo honestamente.
```

🔍 **Predice** · ✏️ **Completa** (andamiaje medio) · 💥 **Trampa** · 🖼️ **Fichas** · ⚡ **Mini-reto**.

> **Datos:** Bank Marketing, igual que la Sesión 5 (45 211 llamadas, 11,7 % "sí").


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"bank.csv": "datos/bank.csv", "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo.
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline(["modelo", "evaluación"], modulo=2, subtitulo="HOY: muchos árboles mejor que uno, y cómo comparar modelos de verdad");

## 1. La Sesión 5 en una celda

Misma preparación: *y* = `deposit`, *one-hot*, `duration` por fuera, separación estratificada. Los árboles **no necesitan escalar**; solo la logística (que usamos como referencia).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

bank = pd.read_csv("bank.csv")
y = (bank["deposit"] == "yes").astype(int)
X = pd.get_dummies(bank.drop(columns=["deposit", "duration"]), drop_first=True, dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

escalador = StandardScaler().fit(X_train)
logistica = LogisticRegression(max_iter=2000).fit(escalador.transform(X_train), y_train)
p_log = logistica.predict_proba(escalador.transform(X_test))[:, 1]

def resumen(nombre, proba, umbral=0.5):
    p = (proba >= umbral).astype(int)
    return {"modelo": nombre, "accuracy": accuracy_score(y_test, p), "recall": recall_score(y_test, p),
            "precisión": precision_score(y_test, p, zero_division=0), "F1": f1_score(y_test, p), "AUC": roc_auc_score(y_test, proba)}

tabla = [resumen("logística (S5)", p_log)]
pd.DataFrame(tabla).set_index("modelo").round(3)

## 2. Dos métricas nuevas: F1 y AUC

En la Sesión 5 vimos que el accuracy no distingue un modelo útil del que nunca dice "sí". Necesitamos números que sí lo hagan:

![F1 y AUC](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S06/img/f1_y_auc.png)

La **curva ROC** sale directo de la figura del umbral de la Sesión 5: cada umbral es un punto.

In [ ]:
fg.roc_explicada(y_test, p_log);

**Lectura:** la logística tiene **AUC ≈ 0,77**: si tomamos al azar un cliente que dijo sí y uno que dijo no, el 77 % de las veces el modelo le da más probabilidad al que dijo sí. El azar daría 50 %.

## 3. Un árbol en Bank Marketing

### 🔍 Predice
Un árbol **sin límite de profundidad** sobre estos datos: ¿qué AUC tendrá en **entrenamiento**? ¿Y en **prueba**, comparado con la logística (0,77)?

**Tu predicción:** entrenamiento ____ · prueba ____

In [ ]:
from sklearn.tree import DecisionTreeClassifier

arbol_libre = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print(f"hojas: {arbol_libre.get_n_leaves():,}")
print(f"AUC entrenamiento: {roc_auc_score(y_train, arbol_libre.predict_proba(X_train)[:, 1]):.3f}")
print(f"AUC prueba:        {roc_auc_score(y_test, arbol_libre.predict_proba(X_test)[:, 1]):.3f}")

## 4. El bosque aleatorio: la sabiduría de muchos árboles

Idea: entrenar **cientos de árboles**, cada uno con:
- una **muestra bootstrap** de las filas (sacar *n* filas al azar **con reemplazo**: unas se repiten, otras quedan por fuera), y
- en cada corte, solo un **subconjunto al azar de las variables**.

Cada árbol sobreajusta **a su manera**. Al promediarlos, los errores se cancelan.

In [ ]:
fg.arbol_vs_bosque();

### 📐 Fundamento
$$\hat{p}_{\text{bosque}}(x) = \frac{1}{B}\sum_{b=1}^{B} \hat{p}_b(x)$$

La probabilidad del bosque es el **promedio** de las probabilidades de sus $B$ árboles.

<details><summary><b>Para quien quiera más:</b> por qué promediar reduce el error</summary>

Si $B$ estimaciones tienen varianza $\sigma^2$ y correlación $\rho$ entre ellas, su promedio tiene varianza $\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$. Más árboles ($B$ grande) matan el segundo término; árboles **menos parecidos** entre sí ($\rho$ pequeño) achican el primero. Por eso el bosque sortea filas **y** variables: para que los árboles se parezcan poco.
</details>

![Dentro del .fit() — bosque aleatorio](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S06/img/dentro_del_fit_bosque.png)

### ✏️ Completa — entrene el bosque
300 árboles, cada hoja con al menos 5 clientes (`min_samples_leaf=5`), `random_state=42` y `n_jobs=-1` (usa todos los núcleos del computador).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

bosque = ...   # TODO: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1) entrenado
p_bosque = bosque.predict_proba(X_test)[:, 1]
arbol5 = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train)
p_arbol = arbol5.predict_proba(X_test)[:, 1]
tabla += [resumen("árbol (max_depth=5)", p_arbol), resumen("bosque", p_bosque)]
pd.DataFrame(tabla).set_index("modelo").round(3)

### La perilla: ¿qué pasa al dejar crecer los árboles?

In [ ]:
fg.complejidad_auc(X_train, y_train, X_test, y_test);

In [ ]:
fg.curvas_roc({"árbol (max_depth=5)": p_arbol, "logística": p_log, "bosque": p_bosque}, y_test);

### ✏️ Completa — el bosque balanceado
Con lo de la Sesión 5: entrene el mismo bosque con `class_weight="balanced"` y agréguelo a la tabla. ¿Qué pasa con el recall y el F1? ¿Y con el AUC?

In [ ]:
bosque_bal = ...   # TODO: el mismo bosque con class_weight="balanced"
tabla.append(resumen("bosque balanceado", ...))   # TODO: probabilidades de prueba de bosque_bal
pd.DataFrame(tabla).set_index("modelo").round(3)

## 5. ¿Qué aprendió el bosque?

Un bosque de 300 árboles no se puede dibujar como el árbol de la Sesión 1. Hay dos formas de mirarlo por dentro.

In [ ]:
fg.importancias(bosque, X.columns);

**`poutcome_success`** (dijo sí en la campaña anterior) encabeza, seguida de la edad, el saldo y el **día** del mes. Ojo con la lectura: importancia es *cuánto lo usa el modelo*, no *qué causa* que el cliente compre. Y las variables numéricas con muchos valores (edad, saldo, día) tienden a salir infladas en esta medida.

### Una fila entra, una predicción sale
Un cliente de prueba que **sí** compró, de probabilidad media. ¿Qué opina cada uno de los 300 árboles?

In [ ]:
positivos = np.where(y_test.values == 1)[0]
i = int(positivos[np.argsort(p_bosque[positivos])[len(positivos) // 2]])
fg.bosque_una_fila(bosque, X_test.iloc[i].values, real=1);

Los árboles están **divididos**: la mayoría dice que es poco probable; unos cuantos están seguros de que sí. El promedio (≈0,25) queda por debajo de 0,5, así que con el umbral por defecto este cliente **no** se llamaría; con el umbral 0,2 de la Sesión 5, sí. La dispersión de los árboles es información: dice qué tan segura es la predicción.

## 6. 💥 La variable que lo cambia todo

En la Sesión 5 dejamos por fuera `duration`: la duración de la llamada, en segundos. Veamos qué tan buena es.

### 🔍 Predice
Si agregamos `duration` al bosque, ¿cuánto cree que sube el AUC (hoy ~0,80)?

**Tu predicción:** AUC ≈ ____

In [ ]:
X_dur = pd.get_dummies(bank.drop(columns=["deposit"]), drop_first=True, dtype=int)
Xd_train, Xd_test, _, _ = train_test_split(X_dur, y, test_size=0.2, random_state=42, stratify=y)
bosque_dur = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1).fit(Xd_train, y_train)
p_dur = bosque_dur.predict_proba(Xd_test)[:, 1]
print(f"AUC sin duration: {roc_auc_score(y_test, p_bosque):.3f}")
print(f"AUC con duration: {roc_auc_score(y_test, p_dur):.3f}")
print("variable más importante:", pd.Series(bosque_dur.feature_importances_, index=X_dur.columns).idxmax())

In [ ]:
fg.curvas_roc({"bosque sin duration": p_bosque, "bosque CON duration": p_dur}, y_test);

**AUC 0,93.** De lejos el mejor modelo del módulo, y `duration` es la variable más importante. ¿Lo entregamos al banco?

In [ ]:
fg.distribucion_por_clase(bank.assign(y=y), "duration", "y", unidad=" s");

## 7. 📦 Entrega E2 — su primer modelo supervisado honesto

Sobre **su** dataset (el de E1), en un notebook de Colab:

| # | Sección | Qué debe contener |
|---|---|---|
| 1 | Pregunta y *y* | Recordatorio de E1 · ¿clasificación o regresión? · proporción de clases o rango |
| 2 | Preparación | Columnas que **quitó** por ser del futuro o identificadores (con la razón) · *one-hot* · escalado si el modelo lo necesita |
| 3 | Separación | `train_test_split` con `stratify` (clasificación) y `random_state` · todo `.fit()` de preprocesamiento solo con entrenamiento |
| 4 | **Línea base** | `DummyClassifier` o `DummyRegressor` con su métrica |
| 5 | Un modelo | Cualquiera de M2 (lineal/Ridge/Lasso, logística, k-NN, árbol, bosque), con una figura de lo que aprendió |
| 6 | **Métrica justificada** | ¿Por qué esa métrica y no otra? (desbalance, costo de cada error, unidades). Mínimo dos frases |
| 7 | Lectura honesta | ¿Cuánto le gana a la línea base? ¿Qué **no** se puede concluir? |

Rúbrica (1–4 cada uno): corrección técnica · ausencia de fuga / split correcto · interpretación honesta · claridad.

Plantilla mínima para empezar (cambie los nombres):

In [ ]:
# ---------- PLANTILLA E2 (cambie lo que está en MAYÚSCULAS) ----------
# df = pd.read_csv("/content/drive/MyDrive/SU_CARPETA/SUS_DATOS.csv")
# y = df["SU_Y"]
# X = pd.get_dummies(df.drop(columns=["SU_Y", "COLUMNAS_DEL_FUTURO", "IDENTIFICADORES"]), drop_first=True, dtype=int)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
#
# from sklearn.dummy import DummyClassifier
# base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
# modelo = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1).fit(X_train, y_train)
#
# for nombre, m in [("línea base", base), ("modelo", modelo)]:
#     proba = m.predict_proba(X_test)[:, 1]
#     print(nombre, "F1", round(f1_score(y_test, proba >= 0.5), 3), "AUC", round(roc_auc_score(y_test, proba), 3))

## ⚡ Mini-reto (opcional)

El bosque no tiene `duration`, pero sí tiene `day` y `month` (fecha del último contacto) entre sus variables más importantes. ¿Son legítimas? Pista: piense cuándo se registran esos datos, y si el banco podría decidir *hoy* a quién llamar el próximo martes usando el día del mes. ¿Qué cambiaría si `day` fuera la fecha de la **próxima** llamada y no de la última?

In [ ]:
# Su análisis aquí

## 9. Cierre del Módulo 2

**Dentro del `.fit()` — los cinco modelos supervisados del curso:**

| | Árbol | Lineal | Logística | k-NN | Bosque |
|---|---|---|---|---|---|
| Qué minimiza | Gini | residuales² | log-loss | nada | Gini (por árbol) |
| Cómo lo busca | exhaustiva y voraz | fórmula cerrada o gradiente | iterativo (lbfgs) | guarda y vota | B árboles al azar, promediados |
| Perilla del optimizador | — | η | `max_iter` | — | — |
| Perilla del modelo | `max_depth` | `alpha` | `C` | `k` | `n_estimators`, `min_samples_leaf` |
| ¿Escalar? | no | sí (Ridge/Lasso, gradiente) | sí | **sí** | no |

**Las tres fugas del curso, juntas:**

| Sesión | Variable | Qué copiaba |
|---|---|---|
| S3 | tasa de víctimas por dirección | la *y* de cada fila (estadística con todos los datos) |
| S4 | puntajes por área | la definición de la *y* |
| S6 | `duration` | la consecuencia de la *y* |

**Para la Sesión 7 — Módulo 3, no supervisado:** se acaba la *y*. Sin respuesta correcta, ¿cómo encuentra un modelo grupos en los datos? Empezamos con K-means y DBSCAN.